In [1]:
import numpy as np
import pandas as pd
import duckdb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. DATA PREPARATION (Simulating FlyRank Page Performance)
# ---------------------------------------------------------
np.random.seed(42)
n_samples = 1500

# Generating realistic search intelligence feature data
df = pd.DataFrame({
    'client_id': np.random.randint(100, 150, size=n_samples),
    'page_id': [f"page_{i}" for i in range(n_samples)],
    'prior_clicks': np.random.poisson(lam=120, size=n_samples),
    'recent_clicks': np.random.poisson(lam=100, size=n_samples),
    'prior_impressions': np.random.poisson(lam=2500, size=n_samples),
    'recent_impressions': np.random.poisson(lam=2100, size=n_samples),
    'avg_position_prior': np.random.uniform(1.0, 25.0, size=n_samples),
    'avg_position_recent': np.random.uniform(1.0, 30.0, size=n_samples),
    'content_age_days': np.random.randint(30, 730, size=n_samples)
})

# Feature engineering
df['click_change_pct'] = (df['recent_clicks'] - df['prior_clicks']) / (df['prior_clicks'] + 1)
df['impression_change_pct'] = (df['recent_impressions'] - df['prior_impressions']) / (df['prior_impressions'] + 1)
df['position_drift'] = df['avg_position_recent'] - df['avg_position_prior']

# Target variable: 1 if page traffic dropped > 20% (Needs Refresh), else 0
df['needs_refresh'] = (df['click_change_pct'] < -0.20).astype(int)

print(f"Dataset Created: {len(df)} pages across {df['client_id'].nunique()} clients.")
print(f"Base Rate (Needs Refresh = 1): {df['needs_refresh'].mean():.2%}\n")

# ---------------------------------------------------------
# 2. HONEST GROUPED SPLIT (By Client ID)
# ---------------------------------------------------------
feature_cols = ['prior_clicks', 'prior_impressions', 'avg_position_prior',
                'content_age_days', 'impression_change_pct', 'position_drift']

X = df[feature_cols]
y = df['needs_refresh']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# ---------------------------------------------------------
# 3. BASELINE VS. MACHINE LEARNING MODEL
# ---------------------------------------------------------
# Baseline Model: Simple rule (If position dropped by more than 3 spots, flag refresh)
y_pred_baseline = (X_test['position_drift'] > 3.0).astype(int)

# ML Model: Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred_ml = clf.predict(X_test)
y_proba_ml = clf.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# 4. EVALUATION & COMPARISON
# ---------------------------------------------------------
acc_base = accuracy_score(y_test, y_pred_baseline)
acc_ml = accuracy_score(y_test, y_pred_ml)
auc_ml = roc_auc_score(y_test, y_proba_ml)

print("=== RESULTS COMPARISON ===")
print(f"Baseline Rule Accuracy: {acc_base:.4f}")
print(f"ML Model Accuracy:     {acc_ml:.4f}")
print(f"ML Model ROC-AUC Score: {auc_ml:.4f}\n")

# Feature Importance Analysis
importances = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("=== FEATURE IMPORTANCES ===")
print(importances)

Dataset Created: 1500 pages across 50 clients.
Base Rate (Needs Refresh = 1): 38.60%

=== RESULTS COMPARISON ===
Baseline Rule Accuracy: 0.5148
ML Model Accuracy:     0.7475
ML Model ROC-AUC Score: 0.8036

=== FEATURE IMPORTANCES ===
prior_clicks             0.327097
impression_change_pct    0.138848
position_drift           0.137300
avg_position_prior       0.137127
content_age_days         0.135977
prior_impressions        0.123652
dtype: float64
